# Data Ingestion and Cleaning
This notebook handles the extraction, cleaning, and consolidation of player data from the general, cards, and campaign datasets.

**Note:** The raw source files (`general.csv`, `cards.csv`, and `campaigns.csv`) are not included in this repository due to data privacy/size constraints. Consequently, this notebook cannot be executed locally. This file is provided for code review and logic documentation purposes only.

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Import dataset
df_g = pd.read_csv('general.csv')

## 1. General Data Cleaning
*Initial filtering and datetime formatting.*

In [3]:
#filter out negative values
df_g = df_g[(df_g['Gold'] >= 0) & (df_g['Gems'] >= 0)  & (df_g['TotalPlay'] > 0) & (df_g['TotalWon'] >= 0) & (df_g['TotalLost'] >= 0) & (df_g['AvatarXp'] >= 0)]

# dropping unnecessary columns
df_g = df_g.drop(columns=['TotalGameTime', 'ApplicationVersion', 'AvatarXp']) # "TotalGameTime" column is highly correlated to "TotalPlay"

In [4]:
# format datetime columns
df_g['FirstInstallDate'] = pd.to_datetime(df_g['FirstInstallDate'], errors='coerce')
df_g['LastSeenDate'] = pd.to_datetime(df_g['LastSeenDate'], errors='coerce')

For this analysis, the FTUE is not taken into account. Churn rates on early stages are known to be high and it is another problem out of the scope of this project.

In [ ]:
# Filter out AvatarLevel greater than 220 and lower than 20
upper_bound = 220
lower_bound = 20
df_g = df_g[df_g['AvatarLevel'].between(lower_bound, upper_bound)]

The snapshot can contain very old data about players that have already left the game a long time ago. This project focuses on the users that has been seen in the last 90 days since the snapshot was taken.

In [6]:
current_date = pd.to_datetime('2024-04-13 00:00:00+00:00', errors='coerce') # this is the day the snapshot was taken, so all the dates and days will be relative to this date

# Only keep users who were seen within 90 days of the snapshot
shape_before = df_g.shape[0]
cutoff_limit = current_date - pd.Timedelta(days=90)
df_g = df_g[df_g['LastSeenDate'] >= cutoff_limit].copy()
print(shape_before- df_g.shape[0])

999


## 2. Card Collection Processing
*Normalization of card power and collection completion rates.*

Once the general dataframe `df_g` is cleaned. The next step is to transform the card dataframe `df_c` and merge it to the cleaned `df_g`, and perform a final validation.

The first step is to extract data about the card levels and power:

In [ ]:
#import dataframe
df_c = pd.read_csv('cards.csv')

# Remove unnecessary columns
df_c = df_c.drop(columns=['Dups','Converts'], axis=1)
df_c = df_c.drop(df_c[df_c['Type'] == 0].index).reset_index(drop=True)

# transform the dataset in order to get data about the cards for every unique user.
df_c1 = pd.pivot_table(df_c, index=['ID'], columns='Rarity', values='Level', aggfunc='sum').reset_index()
df_c1.fillna(0, inplace=True)

# create new columns to represent the power level of all cards in each rarity
df_c1['common_scr'] = df_c1.COMMON/(29*30)     # 29 common cards in the collection, and max level is 30
df_c1['uncommon_scr'] = df_c1.UNCOMMON/(86*60) # 86 uncommon cards in the collection, and max level is 60
df_c1['rare_scr'] = df_c1.RARE/(89*100)        # 89 rare cards in the collection, and max level is 100
df_c1['epic_scr'] = df_c1.EPIC/(87*150)        # 87 epic cards in the collection, and max level is 150

# create a new overall score for all the cards in the user's collection
df_c1['Card_score'] = (df_c1['common_scr'] * 1 + df_c1['uncommon_scr'] * 1 + df_c1['rare_scr'] * 1 + df_c1['epic_scr'] * 1)

# create a new temporary dataframe with just the necessary columns
df_c_pow = df_c1[['ID','Card_score']].copy()

A new temporary dataframe is created in order to get the cards collected by user. This information will also tell how users card collection compares against their game progression.

In [ ]:
# Transform the dataframe to get the cards owned by rarity, per user.
df_c2 = pd.pivot_table(df_c,index=['ID'], columns='Rarity', values='Base', aggfunc='count').reset_index()
df_c2.fillna(0, inplace=True)

# create new columns to get information about the collection completion
df_c2['common_collected'] = df_c2.COMMON/(29) #29 common cards in the collection
df_c2['uncommon_collected'] = df_c2.UNCOMMON/(86) #86 uncommon cards in the collection
df_c2['rare_collected'] = df_c2.RARE/(89) #89 rare cards in the collection
df_c2['epic_collected'] = df_c2.EPIC/(87) #87 epic cards in the collection

df_c2['Cards_collected'] = (df_c2.common_collected + df_c2.uncommon_collected + df_c2.rare_collected+ df_c2.epic_collected)/4

# create a new temporary dataframe with just the necessary columns.
df_c_coll =df_c2[['ID','Cards_collected']].copy()

Now the two temporary dataframes are merged to create the final dataframe with all cards data per user.

In [9]:
df_cards = df_c_coll.merge(df_c_pow, on='ID', how='left')
df_cards.fillna(0, inplace=True)

And it is then merged to the general dataframe `df_g`.

In [10]:
df_gc= df_g.merge(df_cards, left_on='ID', right_on='ID', how='left')

## 3. Game Progression & Final Consolidation
*Campaign star aggregation and exporting the final dataset.*

The final piece of data necessary to start the analysis is the game progression per user. The corresponding dataframe is loaded, transformed and merged into the `df` that has already all the other information.

In [ ]:
df_p = pd.read_csv('campaigns.csv')

# calculate progression per campaign. 
df_p['progress'] = df_p['stars'] / df_p['max_stars']

# join the difficulty into the campaign name to simplify calculations
df_p['campaign'] = df_p['name'] + '_' + df_p['difficulty'].astype(str)

# transform the dataframe to get the progression per campaign for each unique user
df_p1 = pd.pivot_table(df_p, index=['ID'], columns='campaign', values='progress').reset_index()
df_p1 = df_p1.fillna(0)

# calculate an overall game progression metric
df_p1['Overall_progress'] = df_p1.drop(columns=['ID']).mean(axis=1)

df_p1 = df_p1[['ID','Overall_progress']]

The previous dataframe contains individual and overall campaign progress based on the earned stars. It could be useful to keep the total stars earned per user.

In [12]:

# transform dataframe to calculate the number of stars collected per campaign
df_star = pd.pivot_table(df_p, index=['ID'], columns='campaign', values='stars').reset_index()
df_star = df_star.fillna(0)

# calculate the total stars collected per user
df_star['TotalStars'] = df_star.drop(columns=['ID']).sum(axis=1)
df_p2 = df_star[['ID', 'TotalStars']].copy()

# create a new joined dataframe with both progression and stars collected
df_prog = df_p1.merge(df_p2, on='ID', how='left')
df_prog = df_prog.fillna(0)


The final progress dataframe is merged into the general dataframe. this full dataframe is exported to start the analysis in the next step of the project.

In [13]:
# create the final dataframe with all the data consolidated
df_gcp = df_gc.merge(df_prog, on='ID', how='left')

# Export Dataframe
df_gcp.to_csv('Data/cleaned_df.csv', index=False)